# Probabilistic models

Report section 2.1. Distributions fitted to the data, and the dependence between variables.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
FIGURES = Path("../figures")

rng = np.random.default_rng(42)


Monte Carlo Test

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
N = 100_000

# invoer: vast of verdeling
bouwjaar = rng.uniform(1960, 1980, N)
leeftijd = 2026 - bouwjaar
H   = rng.normal(1.5, 0.2, N)                      # kerende hoogte [m]
phi = rng.normal(30, 2, N)                         # hoek inwendige wrijving [graden]
gam = rng.normal(18, 1, N)                         # volumegewicht grond [kN/m3]
q   = rng.normal(5, 1.5, N).clip(0)                # verkeersbelasting [kN/m2]
t0  = rng.normal(0.060, 0.005, N)                  # oorspronkelijke dikte [m]
r   = rng.uniform(0, 0.001, N)                     # afname dikte [m/jaar]
fm  = rng.lognormal(np.log(18_000), 0.2, N)        # buigsterkte [kN/m2]

# formule
Ka = np.tan(np.radians(45 - phi / 2)) ** 2
M_E = Ka * gam * H**3 / 6 + Ka * q * H**2 / 2      # belasting [kNm/m]
t   = (t0 - r * leeftijd).clip(0.001)
M_R = fm * t**2 / 6                                # sterkte [kNm/m]

# resultaat
faalt = M_R < M_E
print("Faalkans:", faalt.mean())

: 

Monte Carlo spreidingsdiagram: elke stip is één trekking. Punten onder de grenslijn M_R = M_E falen.

In [ ]:
# Monte Carlo spreidingsdiagram: belasting M_E tegen sterkte M_R
C_SAFE, C_FAIL, INK, MUTED = "#2a78d6", "#d03b3b", "#0b0b0b", "#52514e"
n_plot = 5_000                                     # subset voor leesbaarheid
idx = rng.choice(N, n_plot, replace=False)
Z = M_R - M_E                                      # veiligheidsmarge [kNm/m]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ok = ~faalt[idx]
ax1.scatter(M_E[idx][ok], M_R[idx][ok], s=8, c=C_SAFE, alpha=0.35, lw=0, label="Veilig (M_R ≥ M_E)")
ax1.scatter(M_E[idx][~ok], M_R[idx][~ok], s=8, c=C_FAIL, alpha=0.35, lw=0, label="Faalt (M_R < M_E)")
lim = max(np.percentile(M_E, 99.9), np.percentile(M_R, 99.9))
ax1.plot([0, lim], [0, lim], c=INK, lw=1.5, ls="--", label="Grenstoestand M_R = M_E")
ax1.set_xlim(0, lim); ax1.set_ylim(0, lim); ax1.set_aspect("equal")
ax1.set_xlabel("Belasting M_E [kNm/m]"); ax1.set_ylabel("Sterkte M_R [kNm/m]")
ax1.set_title(f"Monte Carlo spreiding ({n_plot:,} van {N:,} trekkingen)", loc="left")
ax1.legend(frameon=False, loc="upper left")

bins = np.linspace(np.percentile(Z, 0.1), np.percentile(Z, 99.9), 80)
ax2.hist(Z[Z >= 0], bins=bins, color=C_SAFE, alpha=0.8, label="Veilig")
ax2.hist(Z[Z < 0], bins=bins, color=C_FAIL, alpha=0.8, label="Faalt")
ax2.axvline(0, c=INK, lw=1.5, ls="--")
ax2.set_xlabel("Veiligheidsmarge Z = M_R − M_E [kNm/m]"); ax2.set_ylabel("Aantal trekkingen")
ax2.set_title(f"Faalkans P(Z < 0) = {faalt.mean():.3f}", loc="left")
ax2.legend(frameon=False)

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.2, lw=0.5)
    ax.tick_params(colors=MUTED)

fig.tight_layout()
fig.savefig(FIGURES / "monte_carlo_spreiding.png", dpi=200, bbox_inches="tight")
plt.show()